In [1]:
import os
import time
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from dotenv import load_dotenv

# Vectorization vs Parallelization

For June 2026, the seismic revision routine makes a search of 17 different checks on the seismic data. Each check is a function that takes in the seismic data and performs some operations on it to check for certain conditions. The checks are performed sequentially, which means that each check is performed one after the other. This can be time-consuming, especially if the seismic data is large. In order to handle this, initially parallelization was implemented using the multiprocessing library in Python. This allowed the checks to be performed in parallel, which significantly reduced the time taken to perform the checks. However, this approach had some limitations, such as the overhead of creating and managing multiple processes, and the need to ensure that the checks were thread-safe.

Furthermore, there are some operations that are performed on the seismic data that can be vectorized using libraries such as NumPy. Vectorization allows for the operations to be performed on entire arrays of data at once, rather than iterating through each element individually. This can significantly reduce the time taken to perform the operations, as it takes advantage of the underlying hardware optimizations for array operations. In addition, there is no need to initialize multiple workers or manage the overhead associated with parallelization. By using vectorization, we can reduce energy consumption and improve the efficiency of the seismic revision routine, while also simplifying the code and reducing the potential for errors. Overall, while parallelization can be useful in certain situations, we will check if vectorization is a more efficient and effective approach for handling large datasets and performing complex operations on them.

In this notebook, we will compare the performance of vectorization and parallelization for a specific check in the seismic revision routine. We will implement both approaches and measure the time taken to perform the check on a sample seismic dataset. We will also analyze the results and discuss the advantages and disadvantages of each approach. Finally, we will make recommendations on which approach to use for different scenarios in the seismic revision routine. The main idea is to define single functions to perform each check, and then use either vectorization or parallelization to apply those functions to the seismic data.

## Query seismic data from database

Let's start by querying the seismic data from the database. We will use the `pymysql` library to connect to the database and execute a SQL query to retrieve the seismic data. We will also use the `dotenv` library to load the database credentials from a `.env` file, as stated in the previous notebook.

In [2]:
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

def connect_to_db(
        query: str,
        start_time: dt.datetime = None,
        end_time: dt.datetime = None,
        **kwargs):

    if start_time and end_time:
        start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")
        full_query = f"{query} '{start_time_str}' and '{end_time_str}' ORDER BY Origin.time_value ASC;"  # Filter and order by time_value
    else:
        full_query = query

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        db_connection = pymysql.connect(
            host=os.getenv('SERVER_HOST'),
            user=os.getenv('SERVER_USERNAME'),
            password=os.getenv('SERVER_PASSWORD'),
            db=os.getenv('SERVER_DATABASE')
        )

        try:
            with tqdm(total=1, desc='Querying database...', unit='query', leave=False,bar_format="{desc}") as pbar:
                df = pd.read_sql_query(full_query, db_connection, **kwargs)
                pbar.update(1)
        finally:
            db_connection.close()

    return df

In [3]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

clean_sql = sqlparse.format(revision_query, strip_comments=True).strip()

initial_time = dt.datetime(2023, 5, 3, 0, 0, 0)
final_time = dt.datetime(2026, 3, 17, 0, 0, 0)
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)

print(f"Number of rows in the seismic data: {len(event_df3)}")
# Print how events are by event_type
print("Number of events by event_type:")
print(event_df3['event_type'].value_counts())

Number of rows in the seismic data: 209243
Number of events by event_type:
event_type
not locatable                  116633
earthquake                      78301
not existing                     5211
explosion                        4372
outside of network interest      4200
volcanic eruption                 455
induced earthquake                  3
Name: count, dtype: int64


## 1. Comparison checks

The seismic revision routine performs a list of quality checks on earthquakes, based on relational or absolute thresholds. These checks are designed to identify earthquakes that may not be reliable and may require further investigation. Some of the checks that are performed include:

1. High RMS: This check identifies earthquakes with a high root-mean-square (RMS) value, which indicates that the seismic data is noisy and may not be reliable. The threshold for this check is typically set at a certain value, such as 1.51.
2. Localization uncertainty: This check identifies earthquakes with a high localization uncertainty, which indicates that the location of the earthquake is not well-defined. The threshold for this check is typically set at a certain value, such as 12 km. It is applied both on latitude, longitude, and depth.
3. Depth check: This check identifies earthquakes with a depth that is outside of a certain range, such as between 0 and 700 km. This check is important because earthquakes that are too shallow or too deep may not be reliable and may require further investigation.

For all these type of checks, it is possible to vectorize the solution by applying the check to the entire dataset at once, rather than iterating through each earthquake individually. The idea here is to create a single general function, receiving the filtered seismic data, the threshold value or values (if there are multiple thresholds), and the column to be checked. The function will then apply the check to the entire dataset and return a boolean mask indicating which earthquakes meet the criteria for being flagged as unreliable. This approach can significantly reduce the time taken to perform the checks, without making it too complicated to be debugged or maintained.

In [10]:
# Vectorized implementation of

Time taken for vectorized High RMS check: 0.0947 seconds


,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
65,2023-05-04 14:46:31,SGC2023isxsdy,32.960000,1.085318,1.680000,10.500000,5.444722,5.444722,13.0,13.0,...,NaN,earthquake,SGC,"Zapatoca - Santander, Colombia",6.828333,-73.292667,MLr_vmm,Hypo71,VMM,None
1003,2023-05-21 06:02:55,SGC2023jxlrzj,3.000000,1.458376,1.658857,10.000000,5.868986,5.868986,7.0,7.0,...,NaN,earthquake,SGC,"Tauramena - Casanare, Colombia",5.041500,-72.815833,MLr_3,Hypo71,RSNC,None
1488,2023-05-26 15:21:12,SGC2023khivmr,0.000000,2.543453,1.530328,0.000000,2.467309,1.702655,30.0,30.0,...,15.0,earthquake,SGC,Mar Caribe,8.781057,-77.072632,MLr_4,LOCSAT,iasp91,None
1635,2023-05-29 00:02:46,SGC2023klrorc,42.929688,3.947252,1.799056,7.069558,2.543656,3.040692,139.0,119.0,...,NaN,earthquake,SGC,"Dabeiba - Antioquia, Colombia",6.974757,-76.452419,MLr_1,NonLinLoc,Poveda_et_al_2018,None
1652,2023-05-29 07:32:35,SGC2023kmgmio,1.360000,1.253778,1.550000,15.900000,4.666905,4.666905,10.0,10.0,...,NaN,earthquake,SGC,"Puerto BoyacÃ¡ - BoyacÃ¡, Colombia",5.912833,-74.282167,MLr_vmm,Hypo71,VMM,None
1704,2023-05-30 04:12:21,SGC2023knvopr,0.000000,2.593147,1.934085,0.000000,6.827690,11.838607,9.0,9.0,...,8.0,earthquake,SGC,Sur de Panama,5.444564,-81.825737,MLr,LOCSAT,iasp91,None
1727,2023-05-30 10:12:56,SGC2023kohnis,150.507812,2.494762,1.520386,6.934761,3.300220,5.716334,66.0,66.0,...,NaN,earthquake,SGC,"Los Santos - Santander, Colombia",6.822358,-73.157462,MLr_3,NonLinLoc,Poveda_et_al_2018,None
6888,2023-08-23 09:30:59,SGC2023qogvzi,10.000000,NaN,9.544262,NaN,NaN,NaN,NaN,NaN,...,NaN,earthquake,SGC,"FÃ³meque - Cundinamarca, Colombia",4.475400,-73.859300,None,,,None
8978,2023-09-14 14:25:09,SGC2023sdaouu,13.180000,5.093257,1.520000,2.300000,1.202082,1.202082,102.0,102.0,...,NaN,earthquake,SGC,"El CantÃ³n del San Pablo (ManagrÃº) - ChocÃ³, ...",5.324833,-76.753833,MLr_1,Hypo71,RSNC,b'DESTACADO'
30753,2024-05-21 09:58:15,SGC2024jzpfqz,39.882812,3.938856,1.846648,6.049333,2.715975,2.561944,136.0,121.0,...,NaN,earthquake,SGC,"MurindÃ³ - Antioquia, Colombia",6.765209,-76.714289,MLr_1,NonLinLoc,Poveda_et_al_2018,b'DESTACADO'


In [11]:
# Order dataframe by RMS value in descending order
high_rms_events_sorted = high_rms_events.sort_values(by='quality_standardError', ascending=False).reset_index(drop=True)
high_rms_events_sorted

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
0,2023-08-23 09:30:59,SGC2023qogvzi,10.000000,NaN,9.544262,NaN,NaN,NaN,NaN,NaN,...,NaN,earthquake,SGC,"FÃ³meque - Cundinamarca, Colombia",4.475400,-73.859300,None,,,None
1,2024-09-13 15:15:00,SGC2024sdcfyj,109.000000,5.000000,2.377702,8.683108,9.398828,6.133357,60.0,60.0,...,59.0,earthquake,IGP,"Huancavelica, Peru",-13.610000,-74.860000,mB,LOCSAT,iasp91,b'DESTACADO'
2,2023-05-30 04:12:21,SGC2023knvopr,0.000000,2.593147,1.934085,0.000000,6.827690,11.838607,9.0,9.0,...,8.0,earthquake,SGC,Sur de Panama,5.444564,-81.825737,MLr,LOCSAT,iasp91,None
3,2024-05-21 09:58:15,SGC2024jzpfqz,39.882812,3.938856,1.846648,6.049333,2.715975,2.561944,136.0,121.0,...,NaN,earthquake,SGC,"MurindÃ³ - Antioquia, Colombia",6.765209,-76.714289,MLr_1,NonLinLoc,Poveda_et_al_2018,b'DESTACADO'
4,2023-05-29 00:02:46,SGC2023klrorc,42.929688,3.947252,1.799056,7.069558,2.543656,3.040692,139.0,119.0,...,NaN,earthquake,SGC,"Dabeiba - Antioquia, Colombia",6.974757,-76.452419,MLr_1,NonLinLoc,Poveda_et_al_2018,None
5,2025-10-29 08:19:21,SGC2025vhedge,10.000000,4.801020,1.717966,0.000000,1.935249,2.364565,97.0,85.0,...,66.0,earthquake,SGC,OcÃ©ano PacÃ­fico,3.361495,-82.730011,Mw(mB),LOCSAT,iasp91,None
6,2025-04-26 07:54:34,SGC2025idsfhj,0.000000,4.099074,1.717595,0.000000,3.020026,3.106540,44.0,41.0,...,25.0,earthquake,SGC,OcÃ©ano PacÃ­fico,4.187966,-82.218124,M_Pac,LOCSAT,iasp91,b'DESTACADO'
7,2023-05-04 14:46:31,SGC2023isxsdy,32.960000,1.085318,1.680000,10.500000,5.444722,5.444722,13.0,13.0,...,NaN,earthquake,SGC,"Zapatoca - Santander, Colombia",6.828333,-73.292667,MLr_vmm,Hypo71,VMM,None
8,2023-05-21 06:02:55,SGC2023jxlrzj,3.000000,1.458376,1.658857,10.000000,5.868986,5.868986,7.0,7.0,...,NaN,earthquake,SGC,"Tauramena - Casanare, Colombia",5.041500,-72.815833,MLr_3,Hypo71,RSNC,None
9,2025-08-19 16:02:15,SGC2025qhkxnl,0.000000,2.904271,1.636559,0.000000,3.563018,5.394898,24.0,16.0,...,15.0,earthquake,SGC,OcÃ©ano PacÃ­fico,2.605613,-78.976692,MLr_1,LOCSAT,iasp91,None
